# 03 — Outcome Analysis: Review Completion, Intervention Effectiveness & Intervention Targeting

This notebook analyses outcomes for carers who completed a follow-up review assessment. It examines three questions:

1. **Completion** — do review completion rates vary by situation cluster, and where is the biggest opportunity gap?
2. **Effectiveness** — do carers improve across wellbeing domains, and does this vary by situation cluster?
3. **Targeting** — which interventions are associated with the greatest improvement, and do effects differ across clusters?

**Inputs required:**
- `clustering_kmeans1.csv` — output of `02_clustering_analysis.ipynb`; full dataset with situation cluster assignments
- `Data_Review_Cleaned.csv` — cleaned review assessment data
- `df_interventions` — dataframe containing intervention records, with an ID column (`lookup`) and an `intervention_category` column

**Outputs:**
- `figure_completion_diagnosis.png` — three-panel completion rate diagnostic
- `figure_improvement_rate.png` — improvement rate by cluster with 95% confidence intervals
- `figure_effect_size_heatmap.png` — Cohen's d heatmap across clusters × wellbeing domains
- `heatmap_intervention_vs_score.png` — mean score change by intervention type and domain
- `heatmap_intervention_vs_cluster.png` — mean overall score change by cluster and intervention type

## 1. Imports & Data Loading

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import ttest_1samp

#df_all = the data frame with all features that were transformed, plus the clusters
df_all = pd.read_csv('clustering_kmeans1.csv', dtype={'assessment_date': str})

# Load the review data
df_review = pd.read_csv('Data_Review_Cleaned.csv')

## 2. Link Initial Assessments to Review Data

Flag which carers in the full dataset went on to complete a follow-up review assessment. Approximately 2,000 of the ~11,800 initial assessments have a matched review.

In [ ]:

# Assuming:
# - df_all: full dataset (~11,000) with a 'cluster' column (0–4)
# - df_review: review completers (~2,000), linkable by an ID

# Step 1: Flag who completed the review
df_all['completed_review'] = df_all['ID'].isin(df_review['Assessment ID'])

## 3. Merge Assessments & Calculate Change Scores

Merge the initial and review datasets on carer ID, then compute a change score for each of the nine wellbeing domains (review − initial). A **negative** change indicates improvement (lower score = better outcome on these scales). An overall mean change and a binary improvement flag are also derived.

In [ ]:
# Score columns (same names in both dataframes)
score_cols = [
    'health_wellbeing_score_num',
    'work_education_score_num',
    'finance_score_num',
    'time_out_score_num',
    'family_commitments_score_num',
    'relationships_score_num',
    'home_score_num',
    'diet_score_num',
    'safety_score_num'
]

# Merge on the shared ID (adjust column name to whatever links the two)
# df_initial has: participant_id, cluster, and the 9 score columns
# df_review has: participant_id and the 9 score columns

df_merged = df_all.merge(
    df_review[['Assessment ID'] + score_cols],
    left_on='ID',
    right_on='Assessment ID',
    suffixes=('_initial', '_review')
)

print(f"Matched records: {len(df_merged)}")

In [ ]:
# Calculate change for each domain (review - initial)
# Negative = improvement (score decreased = better outcome)
for col in score_cols:
    change_col = col.replace('_score_num', '_change')
    df_merged[change_col] = df_merged[f'{col}_review'] - df_merged[f'{col}_initial']

change_cols = [col.replace('_score_num', '_change') for col in score_cols]

# Overall mean change across all 9 domains
df_merged['overall_change'] = df_merged[change_cols].mean(axis=1)

# Binary flag: did they improve on average?
df_merged['improved'] = df_merged['overall_change'] > 0

## 4. Apply Cluster Labels

Map numeric situation cluster IDs to the descriptive names derived in `02_clustering_analysis.ipynb`.

In [ ]:
# Replace numeric labels with descriptive names derived from your clustering
cluster_labels = {
    0: 'Younger Carers of Complex Parents',
    1: 'The Broad Middle',
    2: 'Intensive Live-In Carers',
    3: 'Retired Spousal Carers',
    4: 'Long-Duration Parent Carers'
}

# Apply throughout
df_all['cluster_name'] = df_all['situation_cluster'].map(cluster_labels)

## 5. Completion Rate by Cluster

Calculate review completion rates per cluster and estimate the **opportunity gap** — how many additional completions could be achieved if each cluster matched the best-performing group's rate.

In [ ]:
# Inspect what makes each cluster distinct
for cluster_id in sorted(df_all['situation_cluster'].unique()):
    subset = df_all[df_all['situation_cluster'] == cluster_id]
    print(f"\n=== Cluster {cluster_id} ===")
    print(f"N = {len(subset)}")
    # If you have text, show most common terms or situation types
    # If you have numeric features, show means vs overall means
    print(subset.describe())  # or.value_counts() on categorical columns

    # Calculate absolute numbers lost per cluster
completion_rates = df_all.groupby('cluster_name')['completed_review'].agg(
    completed='sum',
    total='count',
    rate='mean'
).sort_values('rate')

overall_rate = df_all['completed_review'].mean()

# KEY METRIC: How many extra completions would we get if each cluster 
# matched the best-performing cluster's rate?
best_rate = completion_rates['rate'].max()

completion_rates['rate_pct'] = (completion_rates['rate'] * 100).round(1)
completion_rates['gap_vs_best'] = best_rate - completion_rates['rate']
completion_rates['potential_extra_completions'] = (
    completion_rates['gap_vs_best'] * completion_rates['total']
).astype(int)

## 6. Domain-Level Change & Effect Sizes

For each cluster × domain combination, compute Cohen's d (one-sample against zero) and a paired t-test to assess whether the mean change is meaningfully different from no change. Effects are categorised as small (|d| ≥ 0.2), moderate (|d| ≥ 0.5), or strong (|d| ≥ 0.5, p < 0.05).

In [ ]:
df_merged['cluster_name'] = df_merged['situation_cluster'].map(cluster_labels)

# Cleaner domain names
domain_labels = {
    'health_wellbeing_change': 'Health & Wellbeing',
    'work_education_change': 'Work & Education',
    'finance_change': 'Finance',
    'time_out_change': 'Time Out',
    'family_commitments_change': 'Family Commitments',
    'relationships_change': 'Relationships',
    'home_change': 'Home',
    'diet_change': 'Diet',
    'safety_change': 'Safety'
}


In [ ]:
def cohens_d_one_sample(x, mu=0):
    """Cohen's d for a one-sample test against mu."""
    return (x.mean() - mu) / x.std()

# --- Per-cluster, per-domain effect sizes ---
effect_size_table = pd.DataFrame(index=cluster_labels.values(), 
                                  columns=[domain_labels[c] for c in change_cols])

significance_table = effect_size_table.copy()

for cluster_id, cluster_name in cluster_labels.items():
    subset = df_merged[df_merged['situation_cluster'] == cluster_id]
    for change_col, domain_name in domain_labels.items():
        values = subset[change_col].dropna()
        if len(values) > 1:
            d = cohens_d_one_sample(values)
            t, p = ttest_1samp(values, 0)
            effect_size_table.loc[cluster_name, domain_name] = round(d, 3)
            # Flag practical significance
            if p < 0.05 and abs(d) >= 0.5:
                significance_table.loc[cluster_name, domain_name] = 'STRONG'
            elif p < 0.05 and abs(d) >= 0.2:
                significance_table.loc[cluster_name, domain_name] = 'MODERATE'
            elif p < 0.05:
                significance_table.loc[cluster_name, domain_name] = 'SMALL'
            else:
                significance_table.loc[cluster_name, domain_name] = 'none'

print("\n=== EFFECT SIZES (Cohen's d) ===")
print("Negative = improvement, |d| > 0.2 small, > 0.5 medium, > 0.8 large")
print(effect_size_table)

print("\n=== PRACTICAL SIGNIFICANCE ===")
print(significance_table)

## 7. Visualisations — Completion & Effectiveness

Uncomment the `savefig` lines to export figures at 300 dpi.

### Figure 1 — Review Completion Diagnostic

Three-panel figure: completion rate by cluster (colour-coded against the overall average), group volume, and potential extra completions if lower-performing clusters matched the best rate.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Panel 1: Completion rate by cluster (sorted, with colour coding)
colors = ['#d32f2f' if r < overall_rate * 100 else '#388e3c' 
          for r in completion_rates['rate_pct']]

axes[0].barh(completion_rates.index, completion_rates['rate_pct'], 
             color=colors, edgecolor='black')
axes[0].axvline(x=overall_rate * 100, color='black', linestyle='--', 
                linewidth=1.5, label=f'Overall ({overall_rate*100:.1f}%)')
axes[0].set_xlabel('Completion Rate (%)', fontsize=11)
axes[0].set_title('Which Groups Complete Reviews?', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=10)

# Add percentage labels
for i, (idx, row) in enumerate(completion_rates.iterrows()):
    axes[0].text(row['rate_pct'] + 0.5, i, f"{row['rate_pct']:.1f}%",
                 va='center', fontsize=10, fontweight='bold')

# Panel 2: Volume per cluster
axes[1].barh(completion_rates.index, completion_rates['total'], 
             color='steelblue', edgecolor='black')
axes[1].set_xlabel('Number of Cases', fontsize=11)
axes[1].set_title('How Many People Are in Each Group?', fontsize=13, fontweight='bold')

# Add count labels
for i, (idx, row) in enumerate(completion_rates.iterrows()):
    axes[1].text(row['total'] + 10, i, f"{int(row['total']):,}",
                 va='center', fontsize=10, fontweight='bold')

# Panel 3: Opportunity (extra completions possible)
axes[2].barh(completion_rates.index, completion_rates['potential_extra_completions'],
             color='darkorange', edgecolor='black')
axes[2].set_xlabel('Potential Extra Completions', fontsize=11)
axes[2].set_title('Where Is the Biggest Opportunity?', fontsize=13, fontweight='bold')

# Add count labels
for i, (idx, row) in enumerate(completion_rates.iterrows()):
    axes[2].text(row['potential_extra_completions'] + 1, i, 
                 f"{int(row['potential_extra_completions']):,}",
                 va='center', fontsize=10, fontweight='bold')

plt.suptitle('Review Completion: Diagnosis & Opportunity', 
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
#fig.savefig('figure3_completion_diagnosis.png', dpi=300, bbox_inches='tight', facecolor='white', edgecolor='none')
plt.show()

### Figure 2 — Improvement Rate by Cluster

Proportion of review completers who showed net improvement (positive mean change across all nine domains), with 95% confidence intervals. Red bars fall below the overall average; green bars exceed it.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# ============================================================
# FIGURE 2: Improvement Rate by Cluster (with 95% CI)
# ============================================================

fig2, ax2 = plt.subplots(figsize=(12, 7))

cluster_improvement = df_merged.groupby('cluster_name')['improved'].agg(['mean', 'count'])
cluster_improvement['se'] = np.sqrt(
    cluster_improvement['mean'] * (1 - cluster_improvement['mean']) / cluster_improvement['count']
)
cluster_improvement = cluster_improvement.sort_values('mean')

overall_rate = df_merged['improved'].mean()

colors = ['#d32f2f' if r < overall_rate else '#388e3c'
          for r in cluster_improvement['mean']]

ax2.barh(cluster_improvement.index, cluster_improvement['mean'] * 100,
         xerr=cluster_improvement['se'] * 100 * 1.96,
         color=colors, edgecolor='black', capsize=5, height=0.6)

ax2.axvline(x=overall_rate * 100, color='black', linestyle='--',
            linewidth=1.5, label=f'Overall Average ({overall_rate * 100:.1f}%)')

# Add percentage labels BEYOND the error bars so they don't overlap
for i, (idx, row) in enumerate(cluster_improvement.iterrows()):
    # Calculate the right edge of the error bar
    ci_upper = (row['mean'] + row['se'] * 1.96) * 100
    ax2.text(ci_upper + 1.5, i, f"{row['mean'] * 100:.1f}%",
             va='center', fontsize=11, fontweight='bold')

ax2.set_xlabel('% Who Improved', fontsize=13)
ax2.set_title('Improvement Rate by Group',
              fontsize=16, fontweight='bold')
ax2.legend(fontsize=12, loc='lower right')
ax2.set_xlim(0, 100)

fig2.tight_layout()
#fig2.savefig('figure2_improvement_rate.png', dpi=300, bbox_inches='tight', facecolor='white', edgecolor='none')

plt.show()



### Figure 3 — Effect Size Heatmap (Cluster × Domain)

Cohen's d for each cluster × domain cell. Green = improvement, red = deterioration. Bold annotations indicate statistically significant effects (p < 0.05).

In [ ]:
# ============================================================
# FIGURE 3: Effect Size Heatmap (Cluster × Domain)
# ============================================================

fig3, ax3 = plt.subplots(figsize=(14, 7))

effect_data = effect_size_table.astype(float)

sns.heatmap(effect_data, annot=True, fmt='.1f', cmap='RdYlGn', center=0,
            ax=ax3, linewidths=0.5, linecolor='white',
            annot_kws={'size': 12, 'fontweight': 'bold'},
            cbar_kws={'label': "Cohen's d (positive = improvement)"})

ax3.set_title('Intervention Effectiveness: Where Are We Making a Difference?',
              fontsize=16, fontweight='bold', pad=15)
ax3.set_ylabel('Situation Type', fontsize=13)
ax3.set_xlabel('Life Domain', fontsize=13)
ax3.set_xticklabels(ax3.get_xticklabels(), rotation=45, ha='right', fontsize=11)
ax3.set_yticklabels(ax3.get_yticklabels(), rotation=0, fontsize=11)

fig3.tight_layout()
#fig3.savefig('figure3_effect_size_heatmap.png', dpi=300, bbox_inches='tight', facecolor='white', edgecolor='none')
plt.show()

## 8. Intervention Analysis

Merge intervention records onto the matched review dataset and examine which intervention types are associated with greater improvement — both across wellbeing domains and within each situation cluster.

**Note:** `df_interventions` must be loaded to run this section. It should contain at minimum an ID column (`lookup`) and an `intervention_category` column. The merge is a left join, so carers without a recorded intervention are retained and then filtered out for the intervention-specific analyses.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

import pandas as pd

# Dataset 1: All 11,000 samples with cluster assignments
#df_all        # columns: [id, situation_cluster,...]

# Dataset 2: ~2,000 people with suggested interventions
df_interventions = pd.read_csv("classified_interventions.csv")  # columns: [id, intervention,...]

# Dataset 3: ~2,000 people with review scores
df_reviews = pd.read_csv("Data_Review_Cleaned.csv")           # columns: [id, review_score,...]

# ---- CLUSTER LABELS ----
cluster_labels = {
    0: 'Complex Parents',
    1: 'The Broad Middle',
    2: 'Round-the-Clock',
    3: 'Retired Spouses',
    4: 'Complex Children'
}

df_merged['situation_cluster'] = df_merged['situation_cluster'].map(cluster_labels)

# Score columns (same names in both dataframes)
score_cols = [
    'health_wellbeing_score_num',
    'work_education_score_num',
    'finance_score_num',
    'time_out_score_num',
    'family_commitments_score_num',
    'relationships_score_num',
    'home_score_num',
    'diet_score_num',
    'safety_score_num'
]

# ---- MERGE ORIGINAL + REVIEW (your existing code) ----
df_merged = df_all.merge(
    df_review[['Assessment ID'] + score_cols],
    left_on='ID',
    right_on='Assessment ID',
    suffixes=('_initial', '_review')
)

print(f"Matched records: {len(df_merged)}")

# Calculate change for each domain (review - initial)
for col in score_cols:
    change_col = col.replace('_score_num', '_change')
    df_merged[change_col] = df_merged[f'{col}_review'] - df_merged[f'{col}_initial']

change_cols = [col.replace('_score_num', '_change') for col in score_cols]

# Overall mean change across all 9 domains
df_merged['overall_change'] = df_merged[change_cols].mean(axis=1)

# Binary flag: did they improve on average?
df_merged['improved'] = df_merged['overall_change'] > 0

# ---- CLUSTER LABELS ----
cluster_labels = {
    0: 'Complex Parents',
    1: 'The Broad Middle',
    2: 'Round-the-Clock',
    3: 'Retired Spouses',
    4: 'Complex Children'
}

df_merged['situation_cluster'] = df_merged['situation_cluster'].map(cluster_labels)

# ---- MERGE IN INTERVENTIONS ----
# Adjust 'ID' and 'intervention' column names to match your intervention dataset
df_merged = df_merged.merge(
    df_interventions,  # your intervention dataframe
    left_on='ID',
    right_on='lookup',     # adjust to whatever the ID column is called in df_interventions
    how='left'
)

# Filter to only those with an intervention
df_with_intervention = df_merged.dropna(subset=['intervention_category'])  # adjust column name
print(f"Records with intervention: {len(df_with_intervention)}")

plt.rcParams.update({'font.size': 14})
# =============================================
# VISUALISATION 2: Heatmap — Score Change by Domain AND Intervention
# =============================================
# Rename change columns for cleaner labels
clean_labels = {col: col.replace('_change', '').replace('_', ' ').title() for col in change_cols}

heatmap_data = (
    df_with_intervention.groupby('intervention_category')[change_cols].mean().rename(columns=clean_labels)
)

plt.figure(figsize=(14, 8))
sns.heatmap(
    heatmap_data,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',       # Red = negative (improvement), Green = positive (worsened)
    center=0,
    linewidths=0.5,
    linecolor='white',
    cbar_kws={'label': 'Mean Score Change (Review − Initial)'}
)

plt.title('Mean Score Change by Intervention & Domain', fontsize=14, fontweight='bold')
plt.xlabel('Domain', fontsize=12)
plt.ylabel('Intervention', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('heatmap_intervention_vs_score.png', dpi=300, bbox_inches='tight')
plt.show()

# =============================================
# VISUALISATION 3: Heatmap — Score Change by Cluster AND Intervention
# =============================================
heatmap_cluster = (
    df_with_intervention.groupby(['situation_cluster', 'intervention_category'])['overall_change'].mean().unstack(fill_value=np.nan)
)

plt.figure(figsize=(14, 8))
sns.heatmap(
    heatmap_cluster,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    center=0,
    linewidths=0.5,
    linecolor='white',
    cbar_kws={'label': 'Mean Overall Score Change'}
)

plt.title('Mean Overall Score Change by Cluster & Intervention', fontsize=14, fontweight='bold')
plt.xlabel('Intervention', fontsize=12)
plt.ylabel('Situation Cluster', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('heatmap_intervention_vs_cluster.png', dpi=300, bbox_inches='tight')
plt.show()

